# Concept Explanations Visualization

This notebook visualizes pre-computed concept-based explanations.

It loads cached artifacts (interpretations, global importances, local importances)
from the `data/` directory and renders them using interpreto's `plot_concepts`.

## Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Configuration: adjust these to select the dataset / method / interpretation
# ---------------------------------------------------------------------------

# Dataset short name: "RT", "GE", "BIOS", "AG", "IMDB", "E", "HE"
DATASET_ABBREV = "BIOS"

# Concept method: "seminmf", "ica", "kmeans", "pca", "svd", "batchtopk_sae", "vanilla_sae", "neurons"
METHOD = "seminmf"

# Interpretation: "topk" or "llm"
INTERPRETATION = "topk"

# Number of concepts ratio (nb_concepts = nb_classes * ratio)
NB_CONCEPTS_RATIO = 3

## Imports and path resolution

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so we can import utils
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import json
import torch
from interpreto import plot_concepts

from utils.data import (
    MODELS_DATASETS,
    ABBREVIATIONS,
    DATASET_CLASSES_NAMES,
    DATASET_CLASSES_SUBSETS,
    get_save_root,
)
from utils.concepts import CONCEPT_METHOD_NAMES

In [ ]:
# Resolve full dataset/model names from abbreviation
dataset_name = next(k for k, v in ABBREVIATIONS["datasets"].items() if v == DATASET_ABBREV)
model_name = next(k for k, v in MODELS_DATASETS.items() if v == dataset_name)

# Classes
classes_names = DATASET_CLASSES_NAMES[dataset_name]
nb_concepts = len(classes_names) * NB_CONCEPTS_RATIO

# Paths
save_root = REPO_ROOT / get_save_root(model_name)
method_dir_name = CONCEPT_METHOD_NAMES[METHOD]
if METHOD == "neurons":
    # NeuronsAs uses hidden_dim as nb_concepts (768 for BERT/RoBERTa)
    concept_dirs = list(save_root.glob(f"concept_models/{method_dir_name}_nc*"))
    assert len(concept_dirs) == 1, f"Expected 1 NeuronsAs dir, found {len(concept_dirs)}: {concept_dirs}"
    concept_dir = concept_dirs[0]
else:
    concept_dir = save_root / f"concept_models/{method_dir_name}_nc{nb_concepts}"

print(f"Dataset: {dataset_name}")
print(f"Model: {model_name}")
print(f"Classes: {classes_names}")
print(f"Concept dir: {concept_dir}")
print(f"Exists: {concept_dir.exists()}")

## Load concept interpretations

In [ ]:
# Load interpretations (topk words or LLM labels)
if INTERPRETATION == "topk":
    # Try the standard filename first, then the BIOS variant
    interp_path = concept_dir / "topk_interpretations.json"
elif INTERPRETATION == "llm":
    interp_path = concept_dir / "llm_interpretations.json"
else:
    raise ValueError(f"Unknown interpretation: {INTERPRETATION}")

assert interp_path.exists(), f"Interpretation file not found: {interp_path}"

with open(interp_path) as f:
    raw_interpretations = json.load(f)

# Convert keys to int. Values are either a list of words (topk) or a single string (llm).
concepts_interpretation: dict[int, str] = {}
for k, v in raw_interpretations.items():
    if isinstance(v, list):
        concepts_interpretation[int(k)] = ", ".join(v)
    else:
        concepts_interpretation[int(k)] = v

print(f"Loaded {len(concepts_interpretation)} concept interpretations from {interp_path.name}")
for idx in list(concepts_interpretation.keys())[:5]:
    print(f"  C{idx}: {concepts_interpretation[idx]}")

## Load global importances

In [ ]:
importances_path = concept_dir / "importances.pt"
assert importances_path.exists(), f"Importances not found: {importances_path}"

raw_importances = torch.load(importances_path, map_location="cpu")

# importances.pt stores a list of tensors (one per sample used for gradient computation).
# Stack and average to get shape (nb_classes, nb_concepts)
global_importances = torch.stack(raw_importances).abs().squeeze().mean(0)

print(f"Global importances shape: {global_importances.shape}")
print(f"  Expected: ({len(classes_names)}, {nb_concepts if METHOD != 'neurons' else '?'})")

## Visualize global concept importances

Interactive plot: click on classes to see their most important concepts.

In [ ]:
# Build labels dict: concept_id -> list of interpretation words
concepts_labels = {}
for k, v in concepts_interpretation.items():
    concepts_labels[k] = v.split(", ") if ", " in v else [v]

plot_concepts(
    classes_names=classes_names,
    concepts_importances=global_importances,
    concepts_labels=concepts_labels,
)